# Sterowanie trajektorią w modelach generatywnych

![Kot i dyfuzja](img/img.png)

W modelach generatywnych (dyfuzyjnych i opartych na algorytmach typu *flow-matching*) obraz nie powstaje w jednym kroku. Proces ten określa **trajektoria** – sekwencja stanów prowadząca od początkowego szumu $x_T$ (*noise*) do gotowego obrazu $x_0$. Każdy krok obliczeniowy aktualizuje dane, bezpośrednio decydując o ostatecznym wyglądzie grafiki.

## Przebieg procesu (Pipeline)

Trajektoria fizycznie realizuje się poprzez wieloetapowy *denoising*. Standardowy schemat tego procesu (*pipeline*) wygląda następująco:

1. **Inicjalizacja:** Proces startuje od losowego *Gaussian noise* $x_T \sim \mathcal{N}(0, \mathbf{I})$. W zadaniach edycyjnych (*image-to-image*) wejściem jest oryginalny obraz zaszumiony do ustalonego poziomu $t < T$.
2. **Conditioning:** Model przyjmuje instrukcję $c$ (np. *embedding* promptu z *text encodera*), która wyznacza kierunek odszumiania w rozkładzie $p_\theta(x_{t-1} | x_t, c)$.
3. **Denoising loop:** Sieć neuronowa $\epsilon_\theta$ szacuje ilość szumu w danych $x_t$, a dedykowany *scheduler* go redukuje. Powstaje ciąg stanów $\{x_T, x_{T-1}, \dots, x_0\}$, tworzący fizyczny kształt trajektorii.
4. **Decoding:** Po zakończeniu pętli, *decoder* (np. model VAE) zamienia skompresowane dane z *latent space* na finalne piksele obrazu.

## Wpływ trajektorii na obraz

Trajektoria wyznacza stany pośrednie, co wprost decyduje o procesie powstawania obrazu:

* **Ogólny zarys:** W początkowych krokach (dla dużych $t$) trajektoria ustala kompozycję, proporcje i układ głównych elementów sceny.
* **Tworzenie detali:** W krokach końcowych ($t \to 0$) algorytm generuje drobne szczegóły. Inna trajektoria przy tym samym szumie $x_T$ i warunku $c$ wygeneruje inne tekstury (np. inny układ porów na skórze czy refleksów światła).
* **Conditioning strength:** Trajektoria określa, jak silnie warunek $c$ wpływa na wynik. W fazie *inference time* kontroluje to parametr *CFG Scale* (Classifier-Free Guidance). Ostateczny szum $\tilde{\epsilon}_\theta$ oblicza się ze wzoru:
  $$\tilde{\epsilon}_\theta(x_t, c) = (1 - w)\epsilon_\theta(x_t) + w\epsilon_\theta(x_t, c)$$
  Waga $w$ określa, jak mocno model podąża za promptem zamiast generować własne, losowe elementy.

## Kształt trajektorii

Z punktu widzenia inżynierii obliczeniowej, trajektoria jest zawsze realizowana w sekwencji dyskretnych kroków. O wydajności i precyzji modelu decyduje jednak fizyczny **kształt ścieżki** łączącej początkowy *noise* z docelowym obrazem.

Rozróżniamy dwa główne podejścia do kształtowania trajektorii:

* **Curved trajectories (nieliniowe):** W klasycznych modelach dyfuzyjnych naturalna ścieżka odszumiania jest skomplikowana i wieloetapowa. Wynika to ze złożoności samych danych – algorytm musi nieustannie korygować kierunek, aby znaleźć kompromis między różnymi informacjami przestrzennymi i semantycznymi. Skrzywiona droga wymusza gęsty podział trasy na dużą liczbę krótkich kroków (najczęściej od 30 do 100). Przekłada się to na wyższy koszt obliczeniowy, dłuższy *inference time* oraz ryzyko powstawania artefaktów wizualnych w miejscach nagłych zmian kierunku.
* **Rectified trajectories (liniowe):** Nowoczesne architektury (oparte m.in. na technikach *Rectified Flow* lub *Flow Matching*) wprowadzają matematyczne prostowanie wektorów. Zamiast przechodzić przez wiele skomplikowanych stanów pośrednich, algorytm wytycza najkrótszą, prostą ścieżkę między szumem $x_T$ a obrazem $x_0$. Prosta droga znacznie zmniejsza ryzyko błędu, co pozwala na wydłużenie pojedynczych kroków i redukcję ich całkowitej liczby. Model może wygenerować obraz o wysokiej jakości w zaledwie kilku iteracjach (często od 1 do 4). Oznacza to bardzo krótki *inference time* oraz lepszą spójność z promptem, ponieważ algorytm zmierza bezpośrednio do wyznaczonego celu.

## Co można osiągnąć za pomocą sterowania trajektorią?

Świadome manipulowanie przebiegiem trajektorii to fundament zaawansowanej pracy z generatywną sztuczną inteligencją. Pozwala ono na osiągnięcie konkretnych rezultatów inżynieryjnych i wizualnych:

* **Few-step generation:** Zastosowanie modeli o wyprostowanych trajektoriach pozwala zredukować liczbę kroków z kilkudziesięciu do kilku, generując obrazy w czasie rzeczywistym przy minimalnym zużyciu zasobów obliczeniowych (VRAM).
* **Inpainting / Image-to-image:** Manipulując długością trajektorii, decydujemy o stopniu modyfikacji obrazu. Krótka trajektoria zachowuje oryginalną kompozycję i tożsamość obiektu (zmieniając np. tylko oświetlenie), podczas gdy długa trajektoria pozwala na całkowitą przebudowę sceny.
* **Separacja zarysu od detali:** Trajektoria wyznacza stany pośrednie. W początkowych krokach ustala układ głównych elementów sceny, a w końcowych ($t \to 0$) generuje drobne szczegóły. Odpowiednie zarządzanie tymi etapami zapobiega zjawisku wizualnego przeładowania (tzw. *overbaking*).
* **Redukcja artefaktów i halucynacji:** Gładka, dobrze ukształtowana ścieżka odszumiania zapobiega błędom strukturalnym. Model rzadziej generuje błędną anatomię (np. niepoprawną liczbę palców) lub niefizyczne połączenia obiektów.

## Narzędzia kontroli: Jak wpływamy na trajektorię?

Kontrola ścieżki odszumiania zachodzi na dwóch głównych etapach życia modelu i wykorzystuje ściśle określone zestawy parametrów:

### 1. Training time (Poziom architektury)
W fazie treningu aktualizowane są wagi modelu $\theta$, aby algorytm popełniał jak najmniejszy błąd przy przewidywaniu szumu.
* **Loss function:** Dane treningowe i funkcję błędu projektuje się tak, aby matematycznie wymusić na sieci uczenie się stabilnych, bezpośrednich i prostych ścieżek odszumiania (prostowanie trajektorii odbywa się właśnie tutaj).
* **Distillation:** Uczenie mniejszego modelu (tzw. ucznia) naśladowania trajektorii większego modelu (nauczyciela), aby przyspieszyć proces i zredukować liczbę wymaganych kroków.

### 2. Inference time (Poziom użytkownika)
W fazie generacji użytkownik zmienia parametry gotowego, zamrożonego modelu. To bezpośrednio modyfikuje przebieg fizycznej trajektorii w czasie rzeczywistym:
* **Scheduler / Sampler:** Decyduje o matematycznym sposobie obliczania kolejnych kroków (np. Euler, DPM++ 2M, DDIM). Niektóre *schedulery* lepiej radzą sobie ze skrzywionymi trajektoriami, inne są zoptymalizowane pod modele wyprostowane.
* **Sampling Steps ($N$):** Określa gęstość podziału trajektorii. Zbyt mała liczba kroków spowoduje niedokładne przejście i rozmyty obraz; zbyt duża wydłuży *inference time* bez widocznego wzrostu jakości.
* **CFG Scale ($w$):** Kontroluje, jak mocno model podąża za instrukcją tekstową. Zwiększenie tej wartości zwęża trajektorię wokół warunku wejściowego $c$, zmuszając model do ścisłego trzymania się promptu kosztem naturalnej różnorodności.
* **Denoising Strength:** Używany przy modyfikacji obrazów. Określa punkt startowy trajektorii na osi czasu. Wartość 0.3 oznacza rozpoczęcie procesu blisko końca (zachowanie struktury), a 1.0 oznacza start od całkowitego *noise'u* $x_T$ (wygenerowanie zupełnie nowego obrazu).

